## Лабораторная работа №7


In [1]:
import re
import string
import warnings
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import pymorphy3
morph = pymorphy3.MorphAnalyzer()

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec

warnings.filterwarnings('ignore')

## 1. Загрузка данных


In [2]:

df = pd.read_csv('dialogues.tsv', sep='\t')
raw_row = df.iloc[5]
raw_dialogue = raw_row['dialogue']
raw_dialogue


'<span class=participant_2>Пользователь 2: Привет!</span><br /><span class=participant_1>Пользователь 1: Привет!</span><br /><span class=participant_1>Пользователь 1: Как дела?)</span><br /><span class=participant_2>Пользователь 2: У меня все замечательно. А твои дела как? Чем занимаешься<br />по жизни?</span><br /><span class=participant_2>Пользователь 2: ?</span><br /><span class=participant_1>Пользователь 1: Рада новому знакомству! У меня тоже всё неплохо. Недавно<br />нашла работу по специальности. Я дизайнер</span><br /><span class=participant_1>Пользователь 1: А ты чем занимаешься?</span><br /><span class=participant_1>Пользователь 1: Ты откуда?</span><br /><span class=participant_2>Пользователь 2: Я продавец, выращиванию овощи и фрукты на даче и продаю<br />их. Я ещё кстати и дачница.</span><br /><span class=participant_2>Пользователь 2: Я с Украины )</span><br /><span class=participant_2>Пользователь 2: А ты откуда ?</span><br /><span class=participant_1>Пользователь 1: Круто! 


## Этап 1. Очистка данных от разметки и шума

In [3]:
def clean_html(html_text: str) -> str:
    soup = BeautifulSoup(html_text, 'lxml')
   
    for br in soup.find_all('br'):
        br.replace_with('\n')
    return soup.get_text(separator=' ') 

def clean_noise(text: str) -> str:
    text = re.sub(r'[ \t]+', ' ', text)   
    text = re.sub(r'\n{2,}', '\n', text)  
    return text.strip()

clean_text = clean_html(raw_dialogue)
clean_text = clean_noise(clean_text)

print(clean_text)

Пользователь 2: Привет! 
 Пользователь 1: Привет! 
 Пользователь 1: Как дела?) 
 Пользователь 2: У меня все замечательно. А твои дела как? Чем занимаешься 
 по жизни? 
 Пользователь 2: ? 
 Пользователь 1: Рада новому знакомству! У меня тоже всё неплохо. Недавно 
 нашла работу по специальности. Я дизайнер 
 Пользователь 1: А ты чем занимаешься? 
 Пользователь 1: Ты откуда? 
 Пользователь 2: Я продавец, выращиванию овощи и фрукты на даче и продаю 
 их. Я ещё кстати и дачница. 
 Пользователь 2: Я с Украины ) 
 Пользователь 2: А ты откуда ? 
 Пользователь 1: Круто! Я из Беларуси, Минск 
 Пользователь 2: Я продавец, выращиваю овощи и фрукты на даче и продаю 
 их. Я ещё кстати и дачница. 
 Пользователь 2: Как зовут тебя ? 
 Пользователь 1: А я мечтаю купить дачу, но пока не могу себе этого 
 позволить( 
 Пользователь 1: Катя, а тебя? 
 Пользователь 2: Ира,приятно познакомится ) 
 Пользователь 2: А я мечтаю жить возле моря. 
 Пользователь 1: Кстати, была в Украине, мне там очень понравилось 



## Этап 2. Восстановление структуры диалога

In [4]:
def extract_dialogue(text: str) -> list[dict]:
 
    pattern = re.compile(
        r'Пользователь\s*(\d+)\s*:\s*(.+?)(?=(?:Пользователь\s*\d+\s*:)|$)',
        re.DOTALL
    )
    matches = pattern.findall(text)

    raw_turns = []
    for speaker_id, utterance in matches:
        utt = re.sub(r'\s+', ' ', utterance).strip()
        if utt:
            raw_turns.append({'speaker': f'Пользователь_{speaker_id}', 'text': utt})

    # Объединяем подряд идущие реплики одного участника
    merged = []
    for turn in raw_turns:
        if merged and merged[-1]['speaker'] == turn['speaker']:
            merged[-1]['text'] += ' ' + turn['text']
        else:
            merged.append({'speaker': turn['speaker'], 'text': turn['text']})

    return merged

dialogue = extract_dialogue(clean_text)

print('Структурированный диалог:')

for turn in dialogue:
    print(f"[{turn['speaker']}]: {turn['text']}")

print(f'Всего реплик (после объединения): {len(dialogue)}')

Структурированный диалог:
[Пользователь_2]: Привет!
[Пользователь_1]: Привет! Как дела?)
[Пользователь_2]: У меня все замечательно. А твои дела как? Чем занимаешься по жизни? ?
[Пользователь_1]: Рада новому знакомству! У меня тоже всё неплохо. Недавно нашла работу по специальности. Я дизайнер А ты чем занимаешься? Ты откуда?
[Пользователь_2]: Я продавец, выращиванию овощи и фрукты на даче и продаю их. Я ещё кстати и дачница. Я с Украины ) А ты откуда ?
[Пользователь_1]: Круто! Я из Беларуси, Минск
[Пользователь_2]: Я продавец, выращиваю овощи и фрукты на даче и продаю их. Я ещё кстати и дачница. Как зовут тебя ?
[Пользователь_1]: А я мечтаю купить дачу, но пока не могу себе этого позволить( Катя, а тебя?
[Пользователь_2]: Ира,приятно познакомится ) А я мечтаю жить возле моря.
[Пользователь_1]: Кстати, была в Украине, мне там очень понравилось . Я вообще люблю путешествовать
[Пользователь_2]: Обожаю море.
[Пользователь_1]: Приятно познакомиться
[Пользователь_2]: В каком городе Украины т

In [5]:
dialogue_df = pd.DataFrame(dialogue)
print(dialogue_df.to_string(index=True))

           speaker                                                                                                                                                                                                                 text
0   Пользователь_2                                                                                                                                                                                                              Привет!
1   Пользователь_1                                                                                                                                                                                                   Привет! Как дела?)
2   Пользователь_2                                                                                                                                                У меня все замечательно. А твои дела как? Чем занимаешься по жизни? ?
3   Пользователь_1                                                      


## Этап 3. Нормализация текста

In [6]:
RUSSIAN_STOPWORDS = set(stopwords.words('russian'))
EXTRA_STOPWORDS = {'это', 'такой', 'вот', 'ну', 'да', 'нет', 'ещё', 'тоже'}
RUSSIAN_STOPWORDS.update(EXTRA_STOPWORDS)

def normalize_text(text: str,
                   remove_digits: bool = True,
                   remove_stopwords: bool = True) -> list[str]:
   
    text = text.lower()
    # Удаляем пунктуацию
    punct_pattern = r'[{}]'.format(re.escape(string.punctuation + '«»—…'))
    text = re.sub(punct_pattern, ' ', text)
    if remove_digits:
        text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text, language='russian')

    if remove_stopwords:
        tokens = [t for t in tokens
                  if t not in RUSSIAN_STOPWORDS and len(t) > 1]
    return tokens

dialogue_df['tokens'] = dialogue_df['text'].apply(normalize_text)

print('Результат токенизации:')
for _, row in dialogue_df.iterrows():
    print(f"[{row['speaker']}]: {row['tokens']}")

Результат токенизации:
[Пользователь_2]: ['привет']
[Пользователь_1]: ['привет', 'дела']
[Пользователь_2]: ['замечательно', 'твои', 'дела', 'занимаешься', 'жизни']
[Пользователь_1]: ['рада', 'новому', 'знакомству', 'всё', 'неплохо', 'недавно', 'нашла', 'работу', 'специальности', 'дизайнер', 'занимаешься', 'откуда']
[Пользователь_2]: ['продавец', 'выращиванию', 'овощи', 'фрукты', 'даче', 'продаю', 'кстати', 'дачница', 'украины', 'откуда']
[Пользователь_1]: ['круто', 'беларуси', 'минск']
[Пользователь_2]: ['продавец', 'выращиваю', 'овощи', 'фрукты', 'даче', 'продаю', 'кстати', 'дачница', 'зовут']
[Пользователь_1]: ['мечтаю', 'купить', 'дачу', 'пока', 'могу', 'позволить', 'катя']
[Пользователь_2]: ['ира', 'приятно', 'познакомится', 'мечтаю', 'жить', 'возле', 'моря']
[Пользователь_1]: ['кстати', 'украине', 'очень', 'понравилось', 'вообще', 'люблю', 'путешествовать']
[Пользователь_2]: ['обожаю', 'море']
[Пользователь_1]: ['приятно', 'познакомиться']
[Пользователь_2]: ['каком', 'городе', 'ук


## Этап 4. Морфологическая обработка (лемматизация + стемминг)

In [7]:
def lemmatize(tokens: list[str]) -> list[str]:
    return [morph.parse(t)[0].normal_form for t in tokens]  # самый вероятный разбор + нач. форма

dialogue_df['lemmas'] = dialogue_df['tokens'].apply(lemmatize)

for _, row in dialogue_df.iterrows():
    print(f"[{row['speaker']}]: {row['lemmas']}")

[Пользователь_2]: ['привет']
[Пользователь_1]: ['привет', 'дело']
[Пользователь_2]: ['замечательный', 'твой', 'дело', 'заниматься', 'жизнь']
[Пользователь_1]: ['рада', 'новый', 'знакомство', 'всё', 'неплохо', 'недавно', 'найти', 'работа', 'специальность', 'дизайнер', 'заниматься', 'откуда']
[Пользователь_2]: ['продавец', 'выращивание', 'овощ', 'фрукт', 'дача', 'продавать', 'кстати', 'дачница', 'украина', 'откуда']
[Пользователь_1]: ['круто', 'беларусь', 'минск']
[Пользователь_2]: ['продавец', 'выращивать', 'овощ', 'фрукт', 'дача', 'продавать', 'кстати', 'дачница', 'звать']
[Пользователь_1]: ['мечтать', 'купить', 'дача', 'пока', 'мочь', 'позволить', 'катя']
[Пользователь_2]: ['ир', 'приятно', 'познакомиться', 'мечтать', 'жить', 'возле', 'море']
[Пользователь_1]: ['кстати', 'украина', 'очень', 'понравиться', 'вообще', 'любить', 'путешествовать']
[Пользователь_2]: ['обожать', 'море']
[Пользователь_1]: ['приятно', 'познакомиться']
[Пользователь_2]: ['какой', 'город', 'украина']
[Пользовате

In [8]:
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer('russian')

def stem(tokens: list[str]) -> list[str]:
    return [stemmer.stem(t) for t in tokens]

dialogue_df['stems'] = dialogue_df['tokens'].apply(stem)

print('После стемминга:')
for _, row in dialogue_df.iterrows():
    print(f"[{row['speaker']}]: {row['stems']}")

После стемминга:
[Пользователь_2]: ['привет']
[Пользователь_1]: ['привет', 'дел']
[Пользователь_2]: ['замечательн', 'тво', 'дел', 'занима', 'жизн']
[Пользователь_1]: ['рад', 'нов', 'знакомств', 'все', 'неплох', 'недавн', 'нашл', 'работ', 'специальн', 'дизайнер', 'занима', 'откуд']
[Пользователь_2]: ['продавец', 'выращиван', 'овощ', 'фрукт', 'дач', 'прода', 'кстат', 'дачниц', 'украин', 'откуд']
[Пользователь_1]: ['крут', 'беларус', 'минск']
[Пользователь_2]: ['продавец', 'выращива', 'овощ', 'фрукт', 'дач', 'прода', 'кстат', 'дачниц', 'зовут']
[Пользователь_1]: ['мечта', 'куп', 'дач', 'пок', 'мог', 'позвол', 'кат']
[Пользователь_2]: ['ир', 'приятн', 'познаком', 'мечта', 'жит', 'возл', 'мор']
[Пользователь_1]: ['кстат', 'украин', 'очен', 'понрав', 'вообщ', 'любл', 'путешествова']
[Пользователь_2]: ['обожа', 'мор']
[Пользователь_1]: ['приятн', 'познаком']
[Пользователь_2]: ['как', 'город', 'украин']
[Пользователь_1]: ['любл', 'мор', 'крым', 'украин']
[Пользователь_2]: ['жду', 'дожд', 'лет'

In [9]:
compare_df = dialogue_df[['speaker', 'text', 'tokens', 'lemmas', 'stems']].copy()
compare_df.columns = ['Участник', 'Исходный текст', 'Токены', 'Леммы', 'Стеммы']
compare_df

,Участник,Исходный текст,Токены,Леммы,Стеммы
0,Пользователь_2,Привет!,[привет],[привет],[привет]
1,Пользователь_1,Привет! Как дела?),"[привет, дела]","[привет, дело]","[привет, дел]"
2,Пользователь_2,У меня все замечательно. А твои дела как? Чем ...,"[замечательно, твои, дела, занимаешься, жизни]","[замечательный, твой, дело, заниматься, жизнь]","[замечательн, тво, дел, занима, жизн]"
3,Пользователь_1,Рада новому знакомству! У меня тоже всё неплох...,"[рада, новому, знакомству, всё, неплохо, недав...","[рада, новый, знакомство, всё, неплохо, недавн...","[рад, нов, знакомств, все, неплох, недавн, наш..."
4,Пользователь_2,"Я продавец, выращиванию овощи и фрукты на даче...","[продавец, выращиванию, овощи, фрукты, даче, п...","[продавец, выращивание, овощ, фрукт, дача, про...","[продавец, выращиван, овощ, фрукт, дач, прода,..."
5,Пользователь_1,"Круто! Я из Беларуси, Минск","[круто, беларуси, минск]","[круто, беларусь, минск]","[крут, беларус, минск]"
6,Пользователь_2,"Я продавец, выращиваю овощи и фрукты на даче и...","[продавец, выращиваю, овощи, фрукты, даче, про...","[продавец, выращивать, овощ, фрукт, дача, прод...","[продавец, выращива, овощ, фрукт, дач, прода, ..."
7,Пользователь_1,"А я мечтаю купить дачу, но пока не могу себе э...","[мечтаю, купить, дачу, пока, могу, позволить, ...","[мечтать, купить, дача, пока, мочь, позволить,...","[мечта, куп, дач, пок, мог, позвол, кат]"
8,Пользователь_2,"Ира,приятно познакомится ) А я мечтаю жить воз...","[ира, приятно, познакомится, мечтаю, жить, воз...","[ир, приятно, познакомиться, мечтать, жить, во...","[ир, приятн, познаком, мечта, жит, возл, мор]"
9,Пользователь_1,"Кстати, была в Украине, мне там очень понравил...","[кстати, украине, очень, понравилось, вообще, ...","[кстати, украина, очень, понравиться, вообще, ...","[кстат, украин, очен, понрав, вообщ, любл, пут..."



## Этап 5. Числовое представление текста

In [10]:
# каждая реплика - строка с леммами
corpus_lemmas = [' '.join(row) for row in dialogue_df['lemmas']]

# весь диалог как один текст 
all_tokens_lemmas = [t for row in dialogue_df['lemmas'] for t in row]

print('Корпус (реплики):')
for i, doc in enumerate(corpus_lemmas):
    print(f'  [{i}] {doc}')

Корпус (реплики):
  [0] привет
  [1] привет дело
  [2] замечательный твой дело заниматься жизнь
  [3] рада новый знакомство всё неплохо недавно найти работа специальность дизайнер заниматься откуда
  [4] продавец выращивание овощ фрукт дача продавать кстати дачница украина откуда
  [5] круто беларусь минск
  [6] продавец выращивать овощ фрукт дача продавать кстати дачница звать
  [7] мечтать купить дача пока мочь позволить катя
  [8] ир приятно познакомиться мечтать жить возле море
  [9] кстати украина очень понравиться вообще любить путешествовать
  [10] обожать море
  [11] приятно познакомиться
  [12] какой город украина
  [13] любить море крым украина
  [14] ждать дождаться лето поехать море стать любить собака
  [15] путешествовать друг машина поэтому побывать хотя любить пеший прогулка
  [16] пеший прогулка здорово
  [17] любить животное кошатница кот вегаснуть черепаха матильда собака
  [18] аж два собака
  [19] здорово звать
  [20] аз герд два девочка дворняжка подобрать улица б

### 5.1 Bag of Words (CountVectorizer)

In [11]:
cv = CountVectorizer()
bow_matrix = cv.fit_transform(corpus_lemmas)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=cv.get_feature_names_out(),
    index=[f'Реплика {i+1}' for i in range(len(corpus_lemmas))]
)
display(bow_df)

,аж,аз,беларусь,бросить,быть,вегаснуть,возле,вообще,вставать,всё,...,твой,удаваться,украина,улица,фрукт,ходить,хотя,часто,черепаха,эльзить
Реплика 1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 3,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
Реплика 4,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
Реплика 5,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0
Реплика 6,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
Реплика 8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 9,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 10,0,0,0,0,0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0


### 5.2 TF-IDF

In [12]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus_lemmas)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf.get_feature_names_out(),
    index=[f'Реплика {i+1}' for i in range(len(corpus_lemmas))]
)
display(tfidf_df)

,аж,аз,беларусь,бросить,быть,вегаснуть,возле,вообще,вставать,всё,...,твой,удаваться,украина,улица,фрукт,ходить,хотя,часто,черепаха,эльзить
Реплика 1,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Реплика 2,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Реплика 3,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,0.478,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Реплика 4,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.296,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Реплика 5,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,0.000,0.000,0.274,0.000,0.324,0.000,0.000,0.000,0.000,0.000
Реплика 6,0.000,0.000,0.577,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Реплика 7,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.337,0.000,0.000,0.000,0.000,0.000
Реплика 8,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Реплика 9,0.000,0.000,0.000,0.000,0.000,0.000,0.41,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Реплика 10,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.400,0.000,0.000,...,0.000,0.000,0.370,0.000,0.000,0.000,0.000,0.000,0.000,0.000


### 5.3 N-граммы (биграммы)

In [13]:
ngram_vec = CountVectorizer(ngram_range=(2, 2))  # биграммы
ngram_matrix = ngram_vec.fit_transform(corpus_lemmas)

ngram_df = pd.DataFrame(
    ngram_matrix.toarray(),
    columns=ngram_vec.get_feature_names_out(),
    index=[f'Реплика {i+1}' for i in range(len(corpus_lemmas))]
)

nonzero_cols = ngram_df.loc[:, (ngram_df != 0).any(axis=0)]
print('Биграммы (N-граммы, n=2):')
if nonzero_cols.empty:
    print('Биграмм не обнаружено (слишком мало слов в репликах).')
else:
    display(nonzero_cols)

Биграммы (N-граммы, n=2):


,аж два,аз герд,беларусь минск,бросить маленький,быть ложиться,вегаснуть черепаха,возле море,вообще любить,вообще рок,вообще часто,...,удаваться побывать,украина откуда,украина очень,улица бросить,фрукт дача,ходить концерт,хотя любить,часто слушать,черепаха матильда,эльзить очень
Реплика 1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 5,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
Реплика 6,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
Реплика 8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 9,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 10,0,0,0,0,0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0


In [14]:
print("Bag of Words (топ-10 слов по частоте)")
top_words = bow_df.sum().sort_values(ascending=False).head(10)
display(pd.DataFrame(top_words, columns=['Частота']).T)

print("\nTF-IDF (только ненулевые значения)")
for i, row in tfidf_df.iterrows():
    nonzero = row[row > 0].sort_values(ascending=False)
    print(f"\n{i}:")
    display(pd.DataFrame(nonzero).T.round(3))

    print("\nN-граммы (биграммы, только встречающиеся)")
nonzero_ngrams = ngram_df.loc[:, (ngram_df > 0).any()].head(10)

display(nonzero_ngrams)

Bag of Words (топ-10 слов по частоте)


,любить,кстати,море,украина,путешествовать,лето,собака,побывать,два,пока
Частота,8,4,4,4,3,3,3,3,3,3



TF-IDF (только ненулевые значения)

Реплика 1:


,привет
Реплика 1,1.0



N-граммы (биграммы, только встречающиеся)

Реплика 2:


...

,замок,красивый,особенно,родиться,германия,джаз,очень,понравиться,рок
Реплика 28,0.363,0.363,0.363,0.363,0.324,0.324,0.296,0.296,0.296



N-граммы (биграммы, только встречающиеся)

Реплика 29:


,быть,вставать,готовить,думать,завтра,кушать,ладный,ложиться,нужно,рано,семья,спать,пока,понравиться
Реплика 29,0.274,0.274,0.274,0.274,0.274,0.274,0.274,0.274,0.274,0.274,0.274,0.274,0.223,0.223



N-граммы (биграммы, только встречающиеся)


,аж два,аз герд,беларусь минск,бросить маленький,быть ложиться,вегаснуть черепаха,возле море,вообще любить,вообще рок,вообще часто,...,удаваться побывать,украина откуда,украина очень,улица бросить,фрукт дача,ходить концерт,хотя любить,часто слушать,черепаха матильда,эльзить очень
Реплика 1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 5,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
Реплика 6,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
Реплика 8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 9,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Реплика 10,0,0,0,0,0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0


### 5.4 Word2Vec

In [15]:
# Для Word2Vec нужен список предложений (токенов)
sentences = list(dialogue_df['lemmas'])
sentences_nonempty = [s for s in sentences if len(s) > 0]

w2v_model = Word2Vec(
    sentences=sentences_nonempty,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    seed=42,
    epochs=50
)

vocab = list(w2v_model.wv.key_to_index.keys())
print(f'Словарь Word2Vec ({len(vocab)} слов): {vocab}\n')

# вектор первого слова
if vocab:
    sample_word = vocab[0]
    print(f'Вектор слова "{sample_word}" (первые 10 измерений):')
    print(w2v_model.wv[sample_word][:10])

Словарь Word2Vec (127 слов): ['любить', 'море', 'украина', 'кстати', 'рок', 'очень', 'вообще', 'путешествовать', 'пока', 'какой', 'лето', 'собака', 'дача', 'побывать', 'здорово', 'два', 'понравиться', 'поехать', 'прогулка', 'подобрать', 'улица', 'бросить', 'маленький', 'пеший', 'заботиться', 'познакомиться', 'приятно', 'мечтать', 'звать', 'дачница', 'продавать', 'фрукт', 'овощ', 'продавец', 'откуда', 'новый', 'жизнь', 'заниматься', 'дело', 'прийтись', 'привет', 'джаз', 'страна', 'слушать', 'германия', 'выращивать', 'думать', 'родиться', 'замок', 'красивый', 'круто', 'беларусь', 'минск', 'рэп', 'особенно', 'быть', 'купить', 'поп', 'мочь', 'позволить', 'катя', 'ир', 'часто', 'ладный', 'путешествие', 'жить', 'выращивание', 'готовить', 'замечательный', 'твой', 'вставать', 'рано', 'рада', 'нужно', 'знакомство', 'всё', 'неплохо', 'недавно', 'найти', 'работа', 'специальность', 'дизайнер', 'завтра', 'спать', 'ложиться', 'нравиться', 'возле', 'кушать', 'животное', 'кошатница', 'кот', 'вегаснуть

### 5.5 Топ-10 важных слов (по TF-IDF)

In [16]:
# Суммируем TF-IDF веса по всем репликам - это рейтинг слов
word_scores = tfidf_df.sum(axis=0).sort_values(ascending=False)

top10 = word_scores.head(10)
top10_df = top10.reset_index()
top10_df.columns = ['Слово', 'Суммарный TF-IDF']
top10_df.index += 1

print('Топ-10 важных слов диалога (TF-IDF):')
display(top10_df)

Топ-10 важных слов диалога (TF-IDF):


,Слово,Суммарный TF-IDF
1,любить,1.941
2,какой,1.786
3,привет,1.707
4,море,1.693
5,украина,1.625
6,здорово,1.473
7,собака,1.172
8,кстати,1.152
9,дело,1.133
10,звать,1.076
